# Pears 8

### Impors

In [ ]:
import os, sys, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings("ignore")

# Append local path and import custom classes
sys.path.append(os.path.abspath('../scripts'))
from spread import SPREAD
from engine import ENGINE
from backtester import BACKTESTER

### Data

In [ ]:
months = [
    "202501", "202502", "202503", "202504", "202505", "202506",
    "202507", "202508", "202509", "202510", "202511", "202512"
]

my_files = [
    [f"../data/processed/audusd_dukascopy_ask_{m}.parquet" for m in months],
    [f"../data/processed/audusd_dukascopy_bid_{m}.parquet" for m in months],
    [f"../data/processed/nzdusd_dukascopy_ask_{m}.parquet" for m in months],
    [f"../data/processed/nzdusd_dukascopy_bid_{m}.parquet" for m in months]
]

### SPREAD

In [ ]:
builder = SPREAD(threshold=1000) 
df = builder.build(my_files)

print(df.head(5))

### ENGINE

In [ ]:
engine = ENGINE(df)
engine.fit_cointegration(z_window=1000)

# Calling with the new dynamic parameters
df_signals = engine.fit_markov_regimes(
    k_regimes=2, 
    scaling=10000, 
    jitter_size=1e-4, 
    random_seed=123
)

print(df_signals.head(5))

### BACKTEST

In [ ]:
# Feed the output of the ENGINE into the BACKTESTER
bt = BACKTESTER(df_signals)

# Run the simulation!
# entry_z=2.0 means we wait for a 2-standard-deviation divergence
df_results = bt.run(
    entry_z=2.0, 
    exit_z=0.0, 
    danger_threshold=0.5, 
    fee_bps=0.5  # Assumes 0.5 basis points crossing cost
)